# 5. Entraînement du modèle YOLOv9-C (`train_dual.py`)

Ce notebook contient la configuration complète pour lancer, suivre et vérifier le **fine-tuning** de **YOLOv9-C** sur les 21 classes de véhicules (`RoadVision_AI/trafic_data/data_1.yaml`) avec :
- **Poids préentraînés :** `yolov9-c.pt`
- **Architecture :** `models/detect/yolov9-c.yaml`
- **Hyperparamètres :** `data/hyps/hyp.scratch-high.yaml`
- **Paramètres d'entraînement :** `--img 640`, `--batch-size 4`, `--epochs 10`

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

YOLO_DIR = Path("../yolov9").resolve()
DATA_YAML = Path("../trafic_data/data_1.yaml").resolve()
TRAIN_SCRIPT = YOLO_DIR / "train_dual.py"
CFG_YAML = YOLO_DIR / "models/detect/yolov9-c.yaml"
HYP_YAML = YOLO_DIR / "data/hyps/hyp.scratch-high.yaml"
WEIGHTS_PT = YOLO_DIR / "yolov9-c.pt"
RUN_NAME = "yolov9c_vehicles_e10"
RUN_DIR = YOLO_DIR / "runs/train" / RUN_NAME

for p in [YOLO_DIR, DATA_YAML, TRAIN_SCRIPT, CFG_YAML, HYP_YAML, WEIGHTS_PT]:
    assert p.exists(), f"Fichier ou dossier manquant : {p}"

HAS_CUDA = torch.cuda.is_available()
DEVICE = "0" if HAS_CUDA else "cpu"
print(f"Device détecté : {DEVICE} ({torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU local'})")

train_cmd = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--workers", "4" if HAS_CUDA else "2",
    "--device", DEVICE,
    "--batch-size", "4",
    "--data", str(DATA_YAML),
    "--img", "640",
    "--cfg", str(CFG_YAML),
    "--weights", str(WEIGHTS_PT),
    "--hyp", str(HYP_YAML),
    "--epochs", "10",
    "--project", str(YOLO_DIR / "runs/train"),
    "--name", RUN_NAME,
    "--exist-ok",
]
print("Commande d'entraînement :\n", " ".join(train_cmd))

In [ ]:
# Lancer l'entraînement sur GPU (ou CPU si aucun GPU disponible)
# Équivalent CLI depuis RoadVision_AI/yolov9 :
# !python train_dual.py --workers 4 --device 0 --batch-size 4 --data ../trafic_data/data_1.yaml --img 640 --cfg models/detect/yolov9-c.yaml --weights yolov9-c.pt --hyp data/hyps/hyp.scratch-high.yaml --epochs 10 --name yolov9c_vehicles_e10 --exist-ok

process = subprocess.Popen(
    train_cmd,
    cwd=str(YOLO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in process.stdout:
    print(line, end="")
process.wait()
print("\nCode de retour train_dual.py :", process.returncode)

In [ ]:
# Suivre les pertes (train / val) et les métriques de validation (Precision, Recall, mAP@0.5, mAP@0.5:0.95)
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print(df.to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))

    for col in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
        if col in df.columns:
            axes[0].plot(df["epoch"], df[col], marker="o", label=col)
    axes[0].set_title("Pertes d'entraînement")
    axes[0].set_xlabel("Époque")
    axes[0].legend()

    for col in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
        if col in df.columns:
            axes[1].plot(df["epoch"], df[col], marker="o", label=col)
    axes[1].set_title("Pertes de validation")
    axes[1].set_xlabel("Époque")
    axes[1].legend()

    for col in ["metrics/precision", "metrics/recall", "metrics/mAP_0.5", "metrics/mAP_0.5:0.95"]:
        if col in df.columns:
            axes[2].plot(df["epoch"], df[col], marker="o", label=col)
    axes[2].set_title("Métriques de validation")
    axes[2].set_xlabel("Époque")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    results_png = RUN_DIR / "results.png"
    if results_png.exists():
        with Image.open(results_png) as img:
            plt.figure(figsize=(14, 7))
            plt.imshow(img)
            plt.axis("off")
            plt.title("Courbes générées par YOLOv9 (results.png)")
            plt.show()
else:
    print(f"results.csv non trouvé ({results_csv}) : exécutez l'entraînement sur GPU pour générer les métriques.")

In [ ]:
# Identifier et vérifier les checkpoints best.pt et last.pt à la fin de l'entraînement
weights_dir = RUN_DIR / "weights"
best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

for label, path in [("best.pt", best_pt), ("last.pt", last_pt)]:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        print(f"[OK] {label} -> {path} ({size_mb:.2f} Mo) | clés={list(ckpt.keys())[:5]}")
    else:
        print(f"[ABSENT] {label} attendu ici : {path}")

### Compréhension théorique : Fine-tuning, époques, batches, hyperparamètres et checkpoints

1. **Pourquoi cette approche correspond à du Transfer Learning par fine-tuning :**
   - Le modèle est initialisé à partir des poids `yolov9-c.pt` préentraînés sur MS COCO (80 classes). Les représentations visuelles apprises (contours, textures, formes génériques d'objets et de véhicules) sont transférées via `intersect_dicts` (1 448 tenseurs sur 1 460), tandis que la tête de détection est adaptée à nos `21` classes.
   - Contrairement au *feature extraction* (où le backbone reste gelé avec `--freeze`), l'entraînement laisse tous les paramètres dégelés (`--freeze 0` par défaut) afin d'ajuster finement l'ensemble du réseau aux caractéristiques de notre dataset routier : c'est du **Transfer Learning par fine-tuning**.

2. **Rôle des époques, des batches, des hyperparamètres et des checkpoints :**
   - **Époques (`--epochs 10`) :** nombre de passages complets sur les `2 297` images d'entraînement, chacun suivi d'une évaluation sur les `298` images de validation.
   - **Batches (`--batch-size 4`) :** taille du mini-lot d'images traité avant chaque mise à jour des poids (`2 297 / 4 ≈ 575` itérations par époque). Un batch de 4 limite l'utilisation de la VRAM GPU.
   - **Hyperparamètres (`hyp.scratch-high.yaml`) :** définissent la dynamique d'apprentissage (`lr0=0.01`, `lrf=0.01`, `momentum=0.937`, `weight_decay=0.0005`, `warmup_epochs=3.0`), les poids des fonctions de perte (`box=7.5`, `cls=0.5`, `dfl=1.5`) et l'augmentation de données (`mosaic=1.0`, `mixup=0.15`, `copy_paste=0.3`).
   - **Checkpoints (`best.pt` et `last.pt`) :**
     - `last.pt` sauvegarde les poids et l'état d'optimisation à la dernière époque effectuée (permet de reprendre avec `--resume`).
     - `best.pt` sauvegarde le modèle ayant obtenu le meilleur score de validation (*fitness*, pondération de `mAP@0.5` et `mAP@0.5:0.95`) ; c'est le checkpoint à retenir pour l'évaluation sur `test` et l'inférence.